# BAA10Y Data Exploration

| Series | Why it's here |
|---|---|
| `BAA10Y` | Moody's Seasoned Baa Corporate Bond Yield Relative to Yield on 10-Year Treasury Constant Maturity  |

**Before running this notebook**, populate the local data cache:

```bash
uv run python scripts/fetch_fred.py
```

After that, no network calls are made — the notebook reads entirely from
the local cache, which is what the `CutoffEnforcer` discipline requires.

In [ ]:
from datetime import datetime
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots

from aieng.forecasting.data import DataService, SeriesMetadata
from aieng.forecasting.data.adapters import StatCanAdapter
from aieng.forecasting.evaluation import ForecastingTask

from data import build_baa10y_target_service

## 1. Build the DataService

In [17]:
svc = build_baa10y_target_service()

## 2. Inspect the registered BAA10Y time series

In [18]:
svc.summary()

,series_id,description,source,units,frequency,n_obs,start,end
0,baa10y_change_1b,BAA10Y cumulative spread change over 1 busines...,"FRED (BAA10Y), derived",percentage-points,B,6911,2000-01-04,2026-06-30
1,baa10y_change_21b,BAA10Y cumulative spread change over 21 busine...,"FRED (BAA10Y), derived",percentage-points,B,6891,2000-02-01,2026-06-30
2,baa10y_change_5b,BAA10Y cumulative spread change over 5 busines...,"FRED (BAA10Y), derived",percentage-points,B,6907,2000-01-10,2026-06-30


## 3. Cutoff filtering

TODO

## 4. Plot the three BAA10Y time series

Code generated by Claude Sonnet 5.

In [ ]:
as_of = "2027-1-1"
series_ids = ["baa10y_change_1b", "baa10y_change_5b", "baa10y_change_21b"]
series_colors = {
    "baa10y_change_1b": "#3987e5",   # blue (dark-mode step)
    "baa10y_change_5b": "#008300",   # green
    "baa10y_change_21b": "#d55181",  # magenta (dark-mode step)
}
series_labels = {
    "baa10y_change_1b": "1 business day",
    "baa10y_change_5b": "5 business days",
    "baa10y_change_21b": "21 business days",
}
recession_periods = [
    ("2001-03-01", "2001-11-01"),
    ("2007-12-01", "2009-06-01"),
    ("2020-02-01", "2020-04-01"),
]

# Each horizon's cumulative change lives on a very different scale (1b is tight
# around zero, 21b swings much wider), so they get their own y-axis per row
# rather than being forced onto one shared scale.
fig = make_subplots(
    rows=len(series_ids),
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.07,
    subplot_titles=[series_labels[sid] for sid in series_ids],
)

for row, series_id in enumerate(series_ids, start=1):
    s = svc.get_series(series_id, as_of=as_of).sort_values("timestamp")
    fig.add_trace(
        go.Scatter(
            x=s["timestamp"],
            y=s["value"],
            mode="lines",
            name=series_labels[series_id],
            line=dict(color=series_colors[series_id], width=1.6),
            showlegend=False,
        ),
        row=row,
        col=1,
    )
    fig.update_yaxes(title_text="Change (pp)", row=row, col=1)

for start, end in recession_periods:
    fig.add_vrect(
        x0=start,
        x1=end,
        fillcolor="#ffffff",
        opacity=0.14,
        line_width=0,
        row="all",
        col=1,
    )

fig.update_xaxes(title_text="Timestamp", row=len(series_ids), col=1)
fig.update_layout(
    title="BAA10Y cumulative spread change by horizon (independent y-scales, shaded: NBER recessions)",
    template="plotly_dark",
    height=700,
)
fig.show()